# Feature Engineering — Field Asset Health Monitor Project

Stage 4 of the pipeline. Consumes `sensor_readings.parquet` + `failure_windows.csv`
(D10 contract) and produces the **model-ready feature table** — one row per time
window, columns = engineered health features — plus its saved artifact.

**Specification (from stage 3's feed-forward):**
- Features in BOTH directions: rising duty/oil (gradual leaks, F4-type) AND
  idle/low-duty indicators (step failures, F1-type).
- Multi-scale look-backs (hours → days; F3's faint signal, if any, is early).
- State-occupancy features (fraction of time per Motor_current mode), not just
  central tendency (6.4).
- Dynamics/variability features — F3's only remaining chance (OQ5).
- Instrument-health features as a separate axis (antiphase, rolling variance).
- Gap-aware: no window may bridge a recording gap.
- Train/eval design per D19: features must carry the label so healthy-only
  training and two-target evaluation are possible downstream.

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from fahm import preprocessing as pp
from fahm import plotting as pl
from fahm import analysis as an
from fahm import features as ft          # NEW package module (see below)

cfg = pp.load_config("../configs/config.yaml")
df = pp.load_processed(cfg)
fw = pd.read_csv(cfg["paths"]["failure_windows"],
                 parse_dates=["start", "end", "maintenance"])
gaps = pp.find_gaps(df, cfg)
labels = an.label_windows(df, fw,
    degraded_periods=[("2020-04-18", "2020-04-30")],
    invalid_periods=[("2020-04-20 04:45", "2020-04-21 01:30")])
print(df.shape, "| labels:", dict(labels.value_counts()))

(1516948, 16) | labels: {'healthy': np.int64(1341222), 'degraded': np.int64(71443), 'prefail': np.int64(59014), 'infail': np.int64(21297), 'postrepair': np.int64(17837), 'invalid': np.int64(6135)}


## 1. Design: the feature grid

Decisions to make and log BEFORE building (D20):
- **Window size** for the feature rows (candidate: 1h — fine enough for
  48h-precursor dynamics, coarse enough that 175 days ≈ 4,200 rows).
- **Gap rule:** windows are computed within contiguous segments only; a
  window crossing a gap > threshold is dropped (stage-1 gap inventory).
- **Look-back scales** for rolling features (candidate: 6h / 24h / 168h).
- **Window label** = majority label of its samples (invalid wins on any
  overlap — trust rule).

### 1.1 Building the window grid

Features are computed per **time window**, not per raw sample — one row of the
feature table summarizes one window. The grid defines those windows.

The one rule that shapes everything: **no window may bridge a recording gap.**
A window spanning a 20-hour hole would average across a discontinuity and
produce meaningless features. So we first split the record into contiguous
**segments** (a new segment begins at every gap > threshold, reusing the
`find_gaps` logic), then tile 1-hour windows *within* each segment. Windows are
therefore 1h from each segment's start — not clock-aligned — a deliberate
consequence of gap-awareness (D20).

In [3]:
grid = ft.build_window_grid(df, cfg)        # window_start | window_end | segment_id

In [4]:
grid

,window_start,window_end,segment_id
0,2020-02-01 00:00:00,2020-02-01 01:00:00,0
1,2020-02-01 01:00:00,2020-02-01 02:00:00,0
2,2020-02-01 02:00:00,2020-02-01 03:00:00,0
3,2020-02-01 03:00:00,2020-02-01 04:00:00,0
4,2020-02-01 04:00:00,2020-02-01 05:00:00,0
...,...,...,...
4055,2020-08-31 15:42:31,2020-08-31 16:42:31,329
4056,2020-08-31 16:42:31,2020-08-31 17:42:31,329
4057,2020-08-31 17:42:31,2020-08-31 18:42:31,329
4058,2020-08-31 18:42:31,2020-08-31 19:42:31,329


### 1.2 Are the windows densely populated?

A 1-hour window holds ~360 samples at the 10s grid — *if* the data is dense.
Windows near sparse stretches could contain too few samples for reliable
features (a std over 5 points is noise). We count actual samples per window
before trusting the grid: a feature built on an under-filled window is a
silent poison.

In [5]:
# how many actual samples fall in each window? flag the thin ones
counts = []
for _, w in grid.iterrows():
    counts.append(an.window_mask(df, w["window_start"], w["window_end"]).sum())
grid["n_samples"] = counts
print(grid["n_samples"].describe())
print("windows with <60 samples (<10 min of data):", (grid["n_samples"] < 60).sum())

count    4060.000000
mean      360.537931
std        12.821256
min       297.000000
25%       363.000000
50%       363.000000
75%       363.000000
max       364.000000
Name: n_samples, dtype: float64
windows with <60 samples (<10 min of data): 0


### 1.3 Labeling each window

Each window inherits a label from its samples (from stage 3's `label_windows`):
the **majority** row-label, with one override — **`invalid` wins any overlap**.
A single frozen-sensor sample corrupts every feature in the window (mean, std,
slope all pulled toward the stuck value), so a window touching the Apr 20
instrument fault is distrusted wholesale rather than majority-voted. This costs
~2 boundary windows and guarantees no partially-frozen window enters training
(D20 trust rule).

In [6]:
grid_labels = ft.label_grid(grid, labels, df)     # majority label per window

In [7]:
display(grid_labels.head(),grid_labels.tail())
display(grid_labels.value_counts())

0    healthy
1    healthy
2    healthy
3    healthy
4    healthy
Name: label, dtype: str

4055    healthy
4056    healthy
4057    healthy
4058    healthy
4059    healthy
Name: label, dtype: str

label
healthy       3586
degraded       190
prefail        160
infail          56
postrepair      48
invalid         20
Name: count, dtype: int64

In [8]:
# the invalid windows should fall in April, inside the degraded span
inval = grid[grid_labels == "invalid"]
print(inval["window_start"].min(), "→", inval["window_start"].max())

2020-04-20 04:36:50 → 2020-04-20 23:36:50


**Grid built and validated.** 4,060 windows across 332 segments (= 331 gaps + 1),
all densely populated (min 297 samples, 0 thin windows). Label balance mirrors
the row level — healthy 88.3%, with degraded > prefail (the 12-day OQ3 span vs
pooled 48h precursors). The 20 `invalid` windows correspond to the ~20.5h Apr 20
fault at 1h resolution, confined to Apr 20–21 as the priority rule requires.

#### GATE: does duty actually have daily / weekly rhythm? (D22)
If flat, calendar features encode a pattern that isn't there — drop the family.

In [13]:
df["hour"] = df[pp.TIMESTAMP].dt.hour
df["dow"] = df[pp.TIMESTAMP].dt.dayofweek

by_hour = df.groupby("hour")["DV_eletric"].mean()
by_dow = df.groupby("dow")["DV_eletric"].mean()
print("duty by hour:\n", by_hour.round(3))
print("\nduty by day-of-week (0=Mon):\n", by_dow.round(3))
print(f"\nhour spread: {by_hour.max() - by_hour.min():.3f} | dow spread: {by_dow.max() - by_dow.min():.3f}")

duty by hour:
 hour
0     0.182
1     0.183
2     0.172
3     0.153
4     0.161
5     0.157
6     0.156
7     0.163
8     0.164
9     0.153
10    0.164
11    0.157
12    0.160
13    0.152
14    0.155
15    0.160
16    0.161
17    0.159
18    0.163
19    0.152
20    0.148
21    0.158
22    0.167
23    0.166
Name: DV_eletric, dtype: float64

duty by day-of-week (0=Mon):
 dow
0    0.146
1    0.133
2    0.193
3    0.138
4    0.139
5    0.191
6    0.190
Name: DV_eletric, dtype: float64

hour spread: 0.035 | dow spread: 0.060


## 2. Feature families — build one, inspect it, then the next

#### Calendar gate (D22) — family DROPPED, one thread opened
- Hour-of-day: flat (spread 0.035, no daytime/rush pattern) — the APU's load
  does not follow passenger-demand time-of-day. Rush-hour detrend hypothesis
  refuted. No hour features.
- Day-of-week: irregular (spread 0.060) — Wed/Sat/Sun elevated ~0.19 vs ~0.14,
  but NOT a weekday/weekend rhythm. Reads as specific operational events on
  particular days, not a cyclic feature.
- Decision: DROP calendar_features (nothing to cyclically encode or detrend).
  Family count 7 -> 6.

In [14]:
by_dow = df[labels=="healthy"].groupby("dow")["DV_eletric"].mean()

In [17]:
print(f"\nduty by day-of-week (0=Mon): {by_dow.round(3)} dow spread: {by_dow.max() - by_dow.min():.3f}")


duty by day-of-week (0=Mon): dow
0    0.130
1    0.134
2    0.188
3    0.143
4    0.123
5    0.116
6    0.168
Name: DV_eletric, dtype: float64 dow spread: 0.072


### OQ6 (resolved, partially) — day-of-week duty
- Original Wed/Sat/Sun elevation was PART confound: Saturday's high duty was
  the degraded/failure periods leaking in (healthy-only: 0.191 → 0.116 — an
  OQ3 fingerprint). But Wed (0.188) and Sun (0.168) stay elevated on
  HEALTHY-only data vs ~0.13 baseline — a real ~0.05 effect, cause unknown
  (scheduled ops on those days?).
- Consequence: minor. Healthy duty has a small day-of-week baseline
  (~0.13–0.19). Too small to justify calendar features (failure signals are
  10-40x larger), but noted as a residual baseline a duty anomaly score could
  in principle correct for. calendar_features stays DROPPED.

### Examine the rest of the families
Discipline: after each family, inspect its NEW columns (describe + a glance)
BEFORE adding the next. A broken family (all-NaN, wrong scale, degenerate) is
caught immediately, not buried among others. The per-failure effect-size
validation is consolidated in §3; the inline check here is basic sanity.

| family | features (examples) | catches |
|---|---|---|
| **duty & state occupancy** | duty; frac time in motor modes | F4 up, F1 down, 6.4's mode story |
| **pressure dynamics** | TP3 idle-decay slope; cycles/hour | the leak physics directly |
| **thermal** | oil median, oil trend, oil-per-duty | F4 ramp, F1 cool-idle |
| **variability (F3's chance)** | rolling std duty/oil; cycle-duration var | erratic-before-failing (OQ5) |
| **cycle frequency** | Lomb-Scargle dominant freq + drift (D21) | leak-driven rhythm change; F3 long shot |
| **instrument health** | antiphase share; stuck-analog flags | Apr 20-type faults; trust mask |

In [ ]:
# duty_state
f_duty_state = ft.duty_state_features(df, grid)
display(f_duty_state.describe())          # sanity: scale sane? any all-NaN column?
# quick glance: f_duty_state.hist(bins=40, figsize=(12,4)) or per-column as needed

In [ ]:
# pressure_dynamics
f_pressure_dynamics = ft.pressure_dynamics_features(df, grid)
display(f_pressure_dynamics.describe())          # sanity: scale sane? any all-NaN column?
# quick glance: f_pressure_dynamics.hist(bins=40, figsize=(12,4)) or per-column as needed

In [ ]:
# thermal
f_thermal = ft.thermal_features(df, grid)
display(f_thermal.describe())          # sanity: scale sane? any all-NaN column?
# quick glance: f_thermal.hist(bins=40, figsize=(12,4)) or per-column as needed

In [ ]:
# variability
f_variability = ft.variability_features(df, grid)
display(f_variability.describe())          # sanity: scale sane? any all-NaN column?
# quick glance: f_variability.hist(bins=40, figsize=(12,4)) or per-column as needed

In [ ]:
# cycle_frequency
f_cycle_frequency = ft.cycle_frequency_features(df, grid)
display(f_cycle_frequency.describe())          # sanity: scale sane? any all-NaN column?
# quick glance: f_cycle_frequency.hist(bins=40, figsize=(12,4)) or per-column as needed

In [ ]:
# instrument_health
f_instrument_health = ft.instrument_health_features(df, grid)
display(f_instrument_health.describe())          # sanity: scale sane? any all-NaN column?
# quick glance: f_instrument_health.hist(bins=40, figsize=(12,4)) or per-column as needed

In [ ]:
# assemble all families onto the grid (build_features runs FAMILIES in order)
feats = ft.build_features(df, grid, cfg)
feats["label"] = grid_labels
print(feats.shape); feats.columns.tolist()

## 3. Feature validation — do engineered features separate better than raw?

The stage-3 effect-size machinery, now pointed at the features:
per-failure effect sizes of each feature family. Success criterion:
- F4/F1 signals at least as strong as raw sensors gave;
- **any F3 signal at all** in the variability family would be new information;
- instrument-health features fire on the Apr 20 window and nowhere unexpected.

In [ ]:
# an.prefail_effect_by_failure(feats_df, fw, feature_cols, grid_labels, hours=48)
# ... and 168h

## 4. Save the feature artifact

`features.parquet`: one row per window — features + label + window bounds.
The single input for stage 5 (modeling). Path in config (D11 convention).

In [ ]:
# path = ft.save_features(feats, cfg)

## Findings feed-forward (to stage 5 — modeling)
- feature table: <n windows x m features>, artifact at <path>
- which families separate, per failure: <...>
- F3 verdict after dynamics features: <detectable / confirmed sudden>
- instrument-health mask coverage: <...>
- training set definition: windows labeled healthy AND instrument-clean